# 📱 Smartphone Addiction Prediction — Full Colab Notebook
**College SI Lab — Machine Learning Project | Author: Reet Advani**

**Dataset:** `Smartphone_Usage_And_Addiction_Analysis.csv` (16 columns, 7,500 rows)

**Target Column:** `addiction_level` (Mild / Moderate / Severe)

**Run cells from top to bottom. Upload the CSV when prompted.**

---
### Notebook Structure
1. Install / Import Libraries
2. Load Dataset
3. Dataset Overview (shape, dtypes, missing values, duplicates)
4. Exploratory Data Analysis (EDA)
5. Statistical Tests — **Confidence Intervals, Z-Test, T-Test**
6. Statistical Tests — **Chi-Square Test**
7. Statistical Tests — **One-Way ANOVA**
8. Data Pre-processing
9. Correlation & Covariance Analysis
10. Train-Test Split
11. Model Training (Logistic Regression, Decision Tree, Random Forest)
12. Model Evaluation & Selection
13. Save Best Model (PKL files)
14. Sample Prediction Test
15. Download PKL Files

## 1. Install / Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import zipfile
import warnings
warnings.filterwarnings('ignore')

from google.colab import files

# --- Statistical tests ---
from scipy import stats
from scipy.stats import (
    chi2_contingency,   # Chi-Square test
    f_oneway,           # One-Way ANOVA
    ttest_ind,          # Independent samples T-Test
    norm                # Normal distribution (Z-score / CI)
)

# --- Scikit-learn ---
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
print('All libraries imported successfully!')

## 2. Load Dataset

In [ ]:
# Upload the CSV file from your computer
uploaded = files.upload()
csv_file = next(name for name in uploaded.keys() if name.lower().endswith('.csv'))
df_raw = pd.read_csv(csv_file)

print(f'File loaded: {csv_file}')
print(f'Shape: {df_raw.shape}')

## 3. Dataset Overview

In [ ]:
print('=== FIRST 5 ROWS ===')
display(df_raw.head())

print('\n=== LAST 5 ROWS ===')
display(df_raw.tail())

print(f'\n=== SHAPE ===')
print(f'Rows: {df_raw.shape[0]}, Columns: {df_raw.shape[1]}')

print('\n=== COLUMN NAMES ===')
print(df_raw.columns.tolist())

print('\n=== DATA TYPES ===')
display(df_raw.dtypes.to_frame('dtype'))

print('\n=== DATASET INFO ===')
df_raw.info()

print('\n=== MISSING VALUES ===')
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
display(pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct}))

print('\n=== DUPLICATE RECORDS ===')
print(f'Duplicate rows: {df_raw.duplicated().sum()}')

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Use rows with valid addiction_level for EDA
df = df_raw.dropna(subset=['addiction_level']).copy()
df.columns = df.columns.str.strip()
df = df.drop_duplicates()

NUMERIC_COLS = [
    'age', 'daily_screen_time_hours', 'social_media_hours',
    'gaming_hours', 'work_study_hours', 'sleep_hours',
    'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time'
]
CATEGORICAL_COLS = ['gender', 'stress_level', 'academic_work_impact']

print('=== NUMERICAL VARIABLES ===')
display(df[NUMERIC_COLS].describe().T.round(2))

print('\n=== CATEGORICAL VARIABLES ===')
for col in CATEGORICAL_COLS:
    print(f'\n{col}:')
    display(df[col].value_counts().to_frame('count'))

In [ ]:
# --- Distribution Plots ---
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
for i, col in enumerate(NUMERIC_COLS):
    sns.histplot(df[col].dropna(), bins=25, kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(f'Distribution: {col}', fontsize=11)
    axes[i].set_xlabel('')
plt.suptitle('Distribution Plots — Numerical Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Box Plots by Addiction Level ---
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
order = ['Mild', 'Moderate', 'Severe']
palette = {'Mild': '#059669', 'Moderate': '#d97706', 'Severe': '#dc2626'}

for i, col in enumerate(NUMERIC_COLS):
    sns.boxplot(data=df, x='addiction_level', y=col, order=order,
                palette=palette, ax=axes[i])
    axes[i].set_title(f'{col} by Addiction Level', fontsize=10)
    axes[i].set_xlabel('')

plt.suptitle('Box Plots — Feature vs Addiction Level', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Scatter Plots ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
pairs = [
    ('daily_screen_time_hours', 'sleep_hours'),
    ('social_media_hours', 'daily_screen_time_hours'),
    ('notifications_per_day', 'app_opens_per_day')
]
for ax, (x_col, y_col) in zip(axes, pairs):
    for lvl, color in palette.items():
        subset = df[df['addiction_level'] == lvl]
        ax.scatter(subset[x_col], subset[y_col], label=lvl,
                   alpha=0.4, s=20, color=color)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(f'{x_col} vs {y_col}')
    ax.legend()
plt.suptitle('Scatter Plots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Statistical Tests — Confidence Intervals, Z-Test & T-Test

In [ ]:
# ============================================================
# 5A. CONFIDENCE INTERVALS (95%) for daily_screen_time_hours
#     per addiction level
# ============================================================
print('=' * 60)
print('95% CONFIDENCE INTERVALS — Daily Screen Time per Group')
print('=' * 60)

ci_results = []
for level in ['Mild', 'Moderate', 'Severe']:
    group_data = df[df['addiction_level'] == level]['daily_screen_time_hours'].dropna()
    n = len(group_data)
    mean = group_data.mean()
    std_err = group_data.std() / np.sqrt(n)          # Standard Error of the Mean
    z_critical = norm.ppf(0.975)                      # 95% CI → z = 1.96
    margin = z_critical * std_err
    lower, upper = mean - margin, mean + margin

    ci_results.append({
        'Addiction Level': level,
        'N': n,
        'Mean (hrs)': round(mean, 4),
        'Std Error': round(std_err, 4),
        'CI Lower (95%)': round(lower, 4),
        'CI Upper (95%)': round(upper, 4)
    })
    print(f'{level:10s}: mean={mean:.4f}, 95% CI=[{lower:.4f}, {upper:.4f}], n={n}')

display(pd.DataFrame(ci_results))

In [ ]:
# ============================================================
# 5B. Z-TEST
#     Compare Severe vs Mild daily screen time
#     (Large sample → use Z-test)
# ============================================================
print('=' * 60)
print('Z-TEST: Severe vs Mild — Daily Screen Time')
print('H0: Mean screen time of Severe == Mean screen time of Mild')
print('H1: Mean screen time of Severe != Mean screen time of Mild')
print('=' * 60)

severe_data = df[df['addiction_level'] == 'Severe']['daily_screen_time_hours'].dropna()
mild_data   = df[df['addiction_level'] == 'Mild']['daily_screen_time_hours'].dropna()

n1, n2 = len(severe_data), len(mild_data)
mean1, mean2 = severe_data.mean(), mild_data.mean()
std1, std2   = severe_data.std(), mild_data.std()

# Z-statistic formula (two independent large samples)
se_diff = np.sqrt((std1**2 / n1) + (std2**2 / n2))
z_stat  = (mean1 - mean2) / se_diff
p_value = 2 * (1 - norm.cdf(abs(z_stat)))  # two-tailed

print(f'Severe: mean={mean1:.4f}, n={n1}, std={std1:.4f}')
print(f'Mild:   mean={mean2:.4f}, n={n2}, std={std2:.4f}')
print(f'Z-statistic : {z_stat:.4f}')
print(f'P-value     : {p_value:.6f}')
print(f'Significance level: 0.05')

if p_value < 0.05:
    print('Result: REJECT H0 — Significant difference in screen time between Severe and Mild groups.')
else:
    print('Result: FAIL TO REJECT H0 — No significant difference detected.')

In [ ]:
# ============================================================
# 5C. INDEPENDENT SAMPLES T-TEST
#     Compare sleep hours: Severe vs Mild
#     H0: Mean sleep hours Severe == Mild
#     H1: Mean sleep hours Severe != Mild
# ============================================================
print('=' * 60)
print('T-TEST (Independent Samples): Severe vs Mild — Sleep Hours')
print('H0: Mean sleep hours of Severe == Mean sleep hours of Mild')
print('H1: Mean sleep hours of Severe != Mean sleep hours of Mild')
print('=' * 60)

sleep_severe = df[df['addiction_level'] == 'Severe']['sleep_hours'].dropna()
sleep_mild   = df[df['addiction_level'] == 'Mild']['sleep_hours'].dropna()

t_stat, p_val = ttest_ind(sleep_severe, sleep_mild, equal_var=False)  # Welch's T-test

print(f'Severe — mean sleep: {sleep_severe.mean():.4f} hrs, n={len(sleep_severe)}')
print(f'Mild   — mean sleep: {sleep_mild.mean():.4f} hrs,   n={len(sleep_mild)}')
print(f'T-statistic : {t_stat:.4f}')
print(f'P-value     : {p_val:.6f}')

if p_val < 0.05:
    print('Result: REJECT H0 — Significant difference in sleep hours between groups.')
else:
    print('Result: FAIL TO REJECT H0 — No significant difference in sleep hours.')

# Visualise
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([sleep_severe, sleep_mild], labels=['Severe', 'Mild'],
           patch_artist=True,
           boxprops=dict(facecolor='#fca5a5'),
           medianprops=dict(color='black', linewidth=2))
ax.set_title(f'Sleep Hours: Severe vs Mild\nT-stat={t_stat:.3f}, p={p_val:.4f}', fontsize=12)
ax.set_ylabel('Sleep Hours')
plt.tight_layout()
plt.show()

## 6. Statistical Tests — Chi-Square Test

In [ ]:
# ============================================================
# CHI-SQUARE TEST
# Test if there is a significant association between
# two categorical variables.
#
# Test 1: stress_level vs addiction_level
# Test 2: academic_work_impact vs addiction_level
# Test 3: gender vs addiction_level
# ============================================================

chi_sq_pairs = [
    ('stress_level',         'addiction_level'),
    ('academic_work_impact', 'addiction_level'),
    ('gender',               'addiction_level'),
]

for col1, col2 in chi_sq_pairs:
    print('=' * 60)
    print(f'CHI-SQUARE TEST: {col1}  vs  {col2}')
    print(f'H0: {col1} and {col2} are INDEPENDENT (no association)')
    print(f'H1: {col1} and {col2} are DEPENDENT (associated)')
    print('-' * 60)

    contingency_table = pd.crosstab(df[col1], df[col2])
    print('Contingency Table:')
    display(contingency_table)

    chi2, p, dof, expected = chi2_contingency(contingency_table)

    # Cramér's V (effect size)
    n = contingency_table.values.sum()
    cramers_v = np.sqrt(chi2 / (n * (min(contingency_table.shape) - 1)))

    print(f'Chi-Square statistic : {chi2:.4f}')
    print(f'Degrees of freedom   : {dof}')
    print(f'P-value              : {p:.6f}')
    print(f"Cramér's V (effect)  : {cramers_v:.4f}")

    if p < 0.05:
        print(f'Result: REJECT H0 — Significant association between {col1} and {col2}.')
    else:
        print(f'Result: FAIL TO REJECT H0 — No significant association detected.')

    # Heatmap of the contingency table
    plt.figure(figsize=(7, 4))
    sns.heatmap(contingency_table, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Contingency Table: {col1} vs {col2}\nχ²={chi2:.2f}, p={p:.4f}', fontsize=11)
    plt.tight_layout()
    plt.show()
    print()

## 7. Statistical Tests — One-Way ANOVA

In [ ]:
# ============================================================
# ONE-WAY ANOVA
# Test if the means of a continuous variable differ
# significantly across the three addiction level groups.
#
# H0: All group means are equal.
# H1: At least one group mean is different.
# ============================================================

anova_features = [
    'daily_screen_time_hours',
    'social_media_hours',
    'sleep_hours',
    'gaming_hours',
    'notifications_per_day',
    'app_opens_per_day'
]

anova_results = []

print('=' * 60)
print('ONE-WAY ANOVA — Comparing group means across Mild / Moderate / Severe')
print('H0: All group means equal  |  H1: At least one group differs')
print('=' * 60)

groups = ['Mild', 'Moderate', 'Severe']

for feat in anova_features:
    group_samples = [df[df['addiction_level'] == g][feat].dropna() for g in groups]
    f_stat, p_val = f_oneway(*group_samples)

    sig = 'YES ***' if p_val < 0.001 else ('YES **' if p_val < 0.01 else ('YES *' if p_val < 0.05 else 'NO'))
    anova_results.append({
        'Feature': feat,
        'F-statistic': round(f_stat, 4),
        'P-value': round(p_val, 6),
        'Significant (α=0.05)': sig,
        'Mild Mean': round(group_samples[0].mean(), 3),
        'Moderate Mean': round(group_samples[1].mean(), 3),
        'Severe Mean': round(group_samples[2].mean(), 3)
    })
    print(f'{feat}: F={f_stat:.4f}, p={p_val:.6f} → {sig}')

print()
display(pd.DataFrame(anova_results))

In [ ]:
# --- ANOVA visualisation: group mean comparison bar chart ---
anova_df = pd.DataFrame(anova_results)
sig_features = anova_df[anova_df['Significant (α=0.05)'].str.startswith('YES')]['Feature'].tolist()

n_sig = len(sig_features)
if n_sig > 0:
    fig, axes = plt.subplots(1, min(n_sig, 3), figsize=(6 * min(n_sig, 3), 5))
    if n_sig == 1:
        axes = [axes]
    palette_anova = ['#059669', '#d97706', '#dc2626']

    for ax, feat in zip(axes, sig_features[:3]):
        means = [df[df['addiction_level'] == g][feat].mean() for g in groups]
        sems  = [df[df['addiction_level'] == g][feat].sem()  for g in groups]
        ax.bar(groups, means, color=palette_anova, edgecolor='white', linewidth=1.5,
               yerr=sems, capsize=5, error_kw={'ecolor': 'black', 'linewidth': 1.5})
        ax.set_title(f'ANOVA: {feat}', fontsize=11)
        ax.set_ylabel('Mean value')
        ax.set_xlabel('Addiction Level')

    plt.suptitle('One-Way ANOVA — Mean Comparison (with SEM bars)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No statistically significant features found in ANOVA.')

## 8. Data Pre-processing

In [ ]:
# Define final feature set and target
FEATURE_COLS = [
    'age', 'gender', 'daily_screen_time_hours', 'social_media_hours',
    'gaming_hours', 'work_study_hours', 'sleep_hours',
    'notifications_per_day', 'app_opens_per_day',
    'weekend_screen_time', 'stress_level', 'academic_work_impact'
]
TARGET_COL = 'addiction_level'

# Excluded to prevent data leakage:
# transaction_id, user_id, addicted_label

df_model = df[FEATURE_COLS + [TARGET_COL]].copy()

# Remove rows with missing target
df_model = df_model.dropna(subset=[TARGET_COL])

print('Final dataset shape for modelling:', df_model.shape)
print('Target distribution:')
display(df_model[TARGET_COL].value_counts().to_frame('count'))

# Numeric vs categorical feature split for the pipeline
NUMERIC_FEATURES = [
    'age', 'daily_screen_time_hours', 'social_media_hours',
    'gaming_hours', 'work_study_hours', 'sleep_hours',
    'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time'
]
CATEGORICAL_FEATURES = ['gender', 'stress_level', 'academic_work_impact']

X = df_model[FEATURE_COLS].copy()
y = df_model[TARGET_COL].copy()

print('\nMissing values per feature:')
display(X.isnull().sum().to_frame('missing'))

## 9. Correlation & Covariance Analysis

In [ ]:
# --- Correlation Matrix ---
print('=== CORRELATION MATRIX (Numeric Features) ===')
corr_matrix = df_model[NUMERIC_FEATURES].corr().round(3)
display(corr_matrix)

plt.figure(figsize=(11, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, center=0, linewidths=0.5)
plt.title('Correlation Heatmap — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Covariance Matrix ---
print('=== COVARIANCE MATRIX (Numeric Features) ===')
cov_matrix = df_model[NUMERIC_FEATURES].cov().round(3)
display(cov_matrix)

plt.figure(figsize=(11, 8))
sns.heatmap(cov_matrix, annot=True, fmt='.2f', cmap='YlOrBr', linewidths=0.5)
plt.title('Covariance Matrix — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Training samples : {len(X_train)}')
print(f'Testing samples  : {len(X_test)}')
print(f'Train ratio      : {len(X_train)/len(X):.0%}')
print(f'Test ratio       : {len(X_test)/len(X):.0%}')
print()
print('Training target distribution:')
display(y_train.value_counts().to_frame('count'))
print('Testing target distribution:')
display(y_test.value_counts().to_frame('count'))

## 11. Model Training — Logistic Regression, Decision Tree, Random Forest

In [ ]:
# --- Build Preprocessing Pipeline ---
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric',      numeric_transformer,      NUMERIC_FEATURES),
    ('categorical',  categorical_transformer,  CATEGORICAL_FEATURES)
])

# --- Define Models ---
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Decision Tree': DecisionTreeClassifier(
        random_state=RANDOM_STATE, class_weight='balanced', max_depth=8
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE,
        class_weight='balanced', max_depth=16, n_jobs=-1
    )
}

# --- Train All Models ---
trained_models = {}
results = []

for model_name, classifier in models.items():
    print(f'Training {model_name}...', end=' ')
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier',   classifier)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    trained_models[model_name] = pipeline
    results.append({
        'Model':            model_name,
        'Accuracy':         accuracy_score(y_test, y_pred),
        'Precision_macro':  precision_score(y_test, y_pred, average='macro', zero_division=0),
        'Recall_macro':     recall_score(y_test, y_pred, average='macro', zero_division=0),
        'F1_macro':         f1_score(y_test, y_pred, average='macro', zero_division=0)
    })
    print('Done.')

results_df = pd.DataFrame(results).sort_values(
    by=['F1_macro', 'Accuracy'], ascending=False
).reset_index(drop=True)

print('\n=== MODEL COMPARISON TABLE ===')
display(results_df.round(4))

## 12. Model Evaluation & Selection

In [ ]:
# --- Select Best Model ---
best_model_name = results_df.loc[0, 'Model']
best_model      = trained_models[best_model_name]
y_pred_best     = best_model.predict(X_test)

print(f'Selected Best Model: {best_model_name}')
print(f'Selection Metric   : Macro F1, then Accuracy')
print()
print('=== CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred_best, zero_division=0))

In [ ]:
# --- Confusion Matrix ---
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    labels=['Mild', 'Moderate', 'Severe'],
    xticks_rotation=20,
    cmap='Blues'
)
plt.title(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Feature Importances (Random Forest only) ---
if best_model_name == 'Random Forest':
    rf = best_model.named_steps['classifier']
    prep = best_model.named_steps['preprocessor']
    feature_names = prep.get_feature_names_out()
    importances = rf.feature_importances_

    fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    fi_df = fi_df.sort_values('Importance', ascending=False).head(12)

    plt.figure(figsize=(10, 5))
    sns.barplot(data=fi_df, x='Importance', y='Feature', palette='viridis')
    plt.title('Top Feature Importances — Random Forest', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 13. Save Best Model (PKL Files)

In [ ]:
# Save the full preprocessing + classifier pipeline
with open('smartphone_addiction_best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Save exact feature column order
with open('smartphone_addiction_feature_columns.pkl', 'wb') as f:
    pickle.dump(FEATURE_COLS, f)

# Save target class labels
with open('smartphone_addiction_target_classes.pkl', 'wb') as f:
    pickle.dump(sorted(y.unique().tolist()), f)

# Save metadata
metadata = {
    'best_model_name': best_model_name,
    'target_column':   TARGET_COL,
    'feature_columns': FEATURE_COLS,
    'excluded_columns': ['transaction_id', 'user_id', 'addicted_label'],
    'target_classes':  sorted(y.unique().tolist()),
    'selection_metric': 'Macro F1, then Accuracy'
}
with open('smartphone_addiction_model_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

print('Saved:')
print('  smartphone_addiction_best_model.pkl')
print('  smartphone_addiction_feature_columns.pkl')
print('  smartphone_addiction_target_classes.pkl')
print('  smartphone_addiction_model_metadata.pkl')

## 14. Sample Prediction Test

In [ ]:
# Test a sample prediction
sample_user = pd.DataFrame([{
    'age': 21,
    'gender': 'Female',
    'daily_screen_time_hours': 9.5,
    'social_media_hours': 4.5,
    'gaming_hours': 2.0,
    'work_study_hours': 2.0,
    'sleep_hours': 5.0,
    'notifications_per_day': 180,
    'app_opens_per_day': 120,
    'weekend_screen_time': 11.0,
    'stress_level': 'High',
    'academic_work_impact': 'Yes'
}])[FEATURE_COLS]

predicted = best_model.predict(sample_user)[0]
probabilities = best_model.predict_proba(sample_user)[0]
prob_dict = dict(zip(best_model.classes_, probabilities))

print(f'Predicted Addiction Level: {predicted}')
print('Class Probabilities:')
for cls, prob in prob_dict.items():
    print(f'  {cls}: {prob*100:.2f}%')

## 15. Download PKL Files

In [ ]:
# Download all PKL files in one ZIP
pkl_files = [
    'smartphone_addiction_best_model.pkl',
    'smartphone_addiction_feature_columns.pkl',
    'smartphone_addiction_target_classes.pkl',
    'smartphone_addiction_model_metadata.pkl'
]

with zipfile.ZipFile('Smartphone_Addiction_PKL_Files.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in pkl_files:
        zf.write(fname)

files.download('Smartphone_Addiction_PKL_Files.zip')
print('Download started!')